### Config with Pydantic

Pydantic Model

In [1]:
from pydantic import BaseModel

class UserConfig(BaseModel):
    '''A simple config model'''
    
    name: str
    top_k: int = 4
    temperature: float = 0.0
    is_debug: bool = False
    
    
# Create an instance — Pydantic validates and converts
config = UserConfig(
    name='agriVision',
    top_k='10',
    temperature='0.8',
    is_debug='true'
)

print(config)
print()

print(f'name:        {config.name} ({type(config.name).__name__})')
print(f'top_k:       {config.top_k} ({type(config.top_k).__name__})')
print(f'temperature: {config.temperature} ({type(config.temperature).__name__})')
print(f'is_debug:    {config.is_debug} ({type(config.is_debug).__name__})')

name='agriVision' top_k=10 temperature=0.8 is_debug=True

name:        agriVision (str)
top_k:       10 (int)
temperature: 0.8 (float)
is_debug:    True (bool)


Validation Errors


In [2]:
from pydantic import BaseModel, ValidationError


class UserConfig(BaseModel):
    
    name: str
    top_k: int = 4
    temperature: float = 0.0
    is_debuf: bool = False
    

# Case 1 — missing required field
try:
    error1 = UserConfig(top_k=4)
    print('Case 1 passed (unexpected)')
    
except ValidationError as e:
    print('Case 1 — missing required field:')
    print(e)
    print()
    
# Case 2 — wrong type that can't be converted
try:
    bad2 = UserConfig(name='agrivoice', top_k='not-a-number')
    print('Case 2 passed (unexpected)')
except ValidationError as e:
    print('Case 2 — invalid value:')
    print(e)

Case 1 — missing required field:
1 validation error for UserConfig
name
  Field required [type=missing, input_value={'top_k': 4}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing

Case 2 — invalid value:
1 validation error for UserConfig
top_k
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not-a-number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/int_parsing


BaseSettings
* BaseSettings is used to read config from environment variables

In [3]:
from pydantic_settings import BaseSettings


class AppConfig(BaseSettings):
    """Config that reads from env vars automatically."""

    name: str = 'agrivoice'
    top_k: int = 4
    log_level: str = 'INFO'
    is_debug: bool = False


# Create with no arguments — Pydantic looks at env vars
config = AppConfig()

print(config)
print()
print(f'name:      {config.name}')
print(f'top_k:     {config.top_k}')
print(f'log_level: {config.log_level}')
print(f'is_debug:  {config.is_debug}')

name='agrivoice' top_k=4 log_level='INFO' is_debug=False

name:      agrivoice
top_k:     4
log_level: INFO
is_debug:  False


In [4]:
# import os

# # Set an env var BEFORE creating the config
# os.environ['TOP_K'] = '15'
# os.environ['LOG_LEVEL'] = 'DEBUG'

# # Create a fresh config — it picks up the new values
# config2 = AppConfig()

# print(f'top_k:     {config2.top_k} ({type(config2.top_k).__name__})')
# print(f'log_level: {config2.log_level}')

Create a .env File

In [5]:
from pathlib import Path

# Directory to .env file
env_path = Path(r'C:\Users\USER\rag_course\15_pure_python\.env')

# The env file content
env_content = r'''
AGRIVOICE_TOP_K=8
AGRIVOICE_LOG_LEVEL=INFO
AGRIVOICE_IS_DEBUG=true
AGRIVOICE_CHROMA_DIR=C:\Users\USER\rag_course\chroma_db_domain
'''

# Write the file
env_path.write_text(env_content, encoding='utf-8')

print(f'Saved: {env_path}')
print()
print('Contents:')
print(env_content)

Saved: C:\Users\USER\rag_course\15_pure_python\.env

Contents:

AGRIVOICE_TOP_K=8
AGRIVOICE_LOG_LEVEL=INFO
AGRIVOICE_IS_DEBUG=true
AGRIVOICE_CHROMA_DIR=C:\Users\USER\rag_course\chroma_db_domain



Configure Pydantic to Use .env

In [6]:
from pydantic_settings import BaseSettings, SettingsConfigDict

class AppConfig(BaseSettings):
    '''Config that reads from a .env file.'''
    
    name: str = 'agrivoice'
    top_k: int = 4
    log_level: str = 'INFO'
    is_debug: bool = False
    chroma_dir: str = ''
    
    # Tell Pydantic where the .env file is
    model_config = SettingsConfigDict(
        env_prefix='AGRIVOICE_',
        env_file=r'C:\Users\USER\rag_course\15_pure_python\.env',
        env_file_encoding='utf-8',
        extra='ignore',
    )
    
config = AppConfig()

print(f'name: {config.name}')
print(f'top_k:      {config.top_k} ({type(config.top_k).__name__})')
print(f'log_level:  {config.log_level}')
print(f'is_debug:   {config.is_debug} ({type(config.is_debug).__name__})')
print(f'chroma_dir: {config.chroma_dir}')

name: agrivoice
top_k:      8 (int)
log_level:  INFO
is_debug:   True (bool)
chroma_dir: C:\Users\USER\rag_course\chroma_db_domain


In [7]:
import os

# Check if TOP_K exists as an actual env var (which would override .env)
print('TOP_K in os.environ?', 'TOP_K' in os.environ)
if 'TOP_K' in os.environ:
    print(f'  Value: {os.environ["TOP_K"]!r}')

# Also check lowercase
print('top_k in os.environ?', 'top_k' in os.environ)

# Show all env vars starting with T
print()
print('All env vars starting with T:')
for k, v in os.environ.items():
    if k.upper().startswith('T'):
        print(f'  {k}={v}')
        

TOP_K in os.environ? True
  Value: '4'
top_k in os.environ? True

All env vars starting with T:
  TEMP=C:\Users\USER\AppData\Local\Temp
  TMP=C:\Users\USER\AppData\Local\Temp
  TOP_K=4
  TERM=xterm-color


AgriVoiceConfig Class

In [8]:
from pathlib import Path
from pydantic_settings import BaseSettings, SettingsConfigDict


class AgriVoiceConfig(BaseSettings):
    '''Single config object for the entire AgriVoice app.'''
    
    # --- App settings ---
    app_name: str = 'agrivoice'
    environment: str = 'development'
    
    # --- App settings ---
    llm_model: str = 'gpt-4o-mini'
    top_k: int = 4
    temperature: float = 0.0
    
    # --- Paths ---
    project_root: str = r'C:\Users\USER\rag_course'
    chroma_dir: str = ''
    
     # --- Logging ---
    log_level: str = 'INFO'
    
    model_config = SettingsConfigDict(
        env_prefix='AGRIVOICE_',
        env_file=r'C:\Users\USER\rag_course\15_pure_python\.env',
        env_file_encoding='utf-8',
        extra='ignore',
    )
    
    @property
    def root_path(self) -> Path:
        '''Project root'''
        
        return Path(self.project_root)
    
    @property
    def chroma_path(self) -> Path:
        '''Chroma store path '''
        
        if self.chroma_dir:
            return Path(self.chroma_dir)
        
        return self.root_path / 'chroma_db_domain'
    
    @property
    def logs_path(self) -> Path:
        '''Logs folder.'''
        return self.root_path / 'logs'
    
    @property
    def is_production(self) -> bool:
        '''True when running in production'''
        
        return self.environment.lower() == 'production'
    


Save to settings.py

In [9]:
from pathlib import Path

# Where the file will be saved
output_path = Path(r'C:\Users\USER\rag_course\15_pure_python\agrivoice\config\settings.py')

# The full source code
source = '''"""
Application settings for AgriVoice.

One class — AgriVoiceConfig — reads env vars and provides typed access
to every setting and derived path the app needs.
"""

from pathlib import Path
from pydantic_settings import BaseSettings, SettingsConfigDict


class AgriVoiceConfig(BaseSettings):
    """Single config object for the entire AgriVoice app."""

    # App settings
    app_name: str = 'agrivoice'
    environment: str = 'development'

    # Model settings
    llm_model: str = 'gpt-4o-mini'
    embedding_model: str = 'text-embedding-3-small'
    top_k: int = 4
    temperature: float = 0.0

    # Paths
    project_root: str = '.'
    chroma_dir: str = ''

    # Logging
    log_level: str = 'INFO'

    model_config = SettingsConfigDict(
        env_prefix='AGRIVOICE_',
        env_file='.env',
        env_file_encoding='utf-8',
        extra='ignore',
    )

    @property
    def root_path(self) -> Path:
        """Project root as a Path object."""
        return Path(self.project_root)

    @property
    def chroma_path(self) -> Path:
        """Chroma store path — from env var or default."""
        if self.chroma_dir:
            return Path(self.chroma_dir)
        return self.root_path / 'chroma_db_domain'

    @property
    def logs_path(self) -> Path:
        """Logs folder."""
        return self.root_path / 'logs'

    @property
    def is_production(self) -> bool:
        """True when running in production."""
        return self.environment.lower() == 'production'
'''

# Write the file
output_path.write_text(source, encoding='utf-8')

print(f'✅ Saved: {output_path}')
print(f'   Size: {output_path.stat().st_size} bytes')

✅ Saved: C:\Users\USER\rag_course\15_pure_python\agrivoice\config\settings.py
   Size: 1567 bytes
